# Búsqueda semántica sobre un PDF con FAISS

Indexamos el paper *Attention Is All You Need* y comparamos **dos formas de trocearlo**:
una página por documento, o chunks de 1000 caracteres. La diferencia se nota en la calidad
de los resultados.

## Objetivos

1. **Cargar** un PDF directamente desde una URL con `PyPDFLoader`.
2. **Indexar** los documentos en FAISS y **persistir** el índice en disco.
3. **Trocear** en chunks y construir un segundo índice.
4. Ver por qué **indexar por chunks** gana a indexar páginas enteras.

## Qué construimos

```mermaid
flowchart TB
    A["PDF remoto<br/>arxiv 1706.03762"] --> B["PyPDFLoader"]
    B --> C["Documents<br/>1 por página"]

    C -.->|alternativa · no se ejecuta| IDX1[("FAISS · por páginas<br/>data/faiss_attention_pages")]

    C --> SP["RecursiveCharacterTextSplitter<br/>1000 car · overlap 150"]
    SP --> D["Chunks"]
    D --> EMB2["☁️ OpenAI · embeddings<br/>text-embedding-3-small"]
    EMB2 --> IDX2[("FAISS · por chunks<br/>data/faiss_attention")]

    IDX2 --> Q2["similarity_search · k=2"]

    classDef openai fill:#10a37f,stroke:#0b6e55,color:#ffffff,stroke-width:2px
    classDef opcional stroke-dasharray: 5 5,color:#888888
    class EMB2 openai
    class IDX1 opcional
```

**Por qué chunks y no páginas:** una página entera produce un embedding "promediado" que
diluye los detalles. Al buscar *"how many attention heads"*, un índice por páginas devolvería
la página que habla de arquitectura en general; el de chunks devuelve el párrafo exacto.
La vía de las páginas queda documentada en la sección 4, en gris en el diagrama, pero no se
construye: el notebook ejecuta solo el camino de chunks.


## 1 · Setup

Imports y API keys desde el `.env` de la raíz del repo.


In [40]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import requests

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings


In [34]:
# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
EMBEDDINGS = OpenAIEmbeddings(model="text-embedding-3-small")

## 2 · Cargar el PDF

`PyPDFLoader` acepta una URL directamente: se descarga el PDF a un temporal y lo parsea.
Devuelve **un `Document` por página**, con `metadata["page"]` para poder citar después.


In [35]:
file_path = "https://arxiv.org/pdf/1706.03762"
docs = PyPDFLoader(file_path).load()

## 3 · Crear o reutilizar el índice

Calcular embeddings cuesta dinero y tiempo, así que no queremos rehacerlos en cada ejecución.
Este helper persiste el índice en disco y lo reutiliza si ya existe.

`save_local()` genera dos archivos: `index.faiss` con los vectores e `index.pkl` con los
textos y su metadata.

> `allow_dangerous_deserialization=True` hace falta porque el `.pkl` se carga con `pickle`.
> Es seguro con índices que has generado tú; nunca lo uses con un índice de origen desconocido.


In [36]:
def load_or_create_index(documents, path, embeddings=EMBEDDINGS):
    """Carga el índice FAISS desde disco si existe; si no, lo crea y lo guarda."""
    path = Path(path)

    # save_local genera index.faiss (vectores) e index.pkl (textos + metadata)
    if (path / "index.faiss").exists() and (path / "index.pkl").exists():
        print(f"Cargando índice existente: {path}")
        return FAISS.load_local(str(path), embeddings, allow_dangerous_deserialization=True)

    print(f"Creando índice ({len(documents)} documentos): {path}")
    index = FAISS.from_documents(documents, embeddings)
    index.save_local(str(path))
    return index

## 4 · Alternativa: indexar por páginas

El notebook funciona sin este paso. Se deja documentado para mostrar que el **mismo helper**
sirve para indexar los documentos sin trocear: un vector por página completa.

```python
faiss_index = load_or_create_index(docs, "data/faiss_attention_pages")
results = faiss_index.similarity_search("attention", k=2)
```

No lo ejecutamos porque construiría un segundo índice —con su coste en embeddings— solo
para ilustrar el contraste. La idea que importa: una página entera produce un embedding
"promediado" que diluye los detalles, así que al preguntar por algo concreto devuelve la
página que trata el tema en general, no el párrafo exacto. Por eso seguimos con chunks.


## 5 · Índice por chunks

Segundo enfoque: trocear cada página en fragmentos de 1000 caracteres, con 150 de solape
para que una frase que caiga en el corte no se pierda entre dos chunks.

Compara el resultado de esta búsqueda con la anterior: la misma pregunta devuelve
fragmentos mucho más precisos.


In [41]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(len(docs), "páginas →", len(chunks), "chunks")

15 páginas → 49 chunks


In [42]:
# Índice por chunks
faiss_chunks = load_or_create_index(chunks, path="data/faiss_attention")
for doc in faiss_chunks.similarity_search("How many attention heads does the base model use?", k=2):
    print(str(doc.metadata["page"]) + ":", doc.page_content[:300], "\n---")


Creando índice (49 documentos): data/faiss_attention
4: output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhib 
---
8: big 6 1024 4096 16 0.3 300K 4.33 26.4 213
development set, newstest2013. We used beam search as described in the previous section, but no
checkpoint averaging. We present these results in Table 3.
In Table 3 rows (A), we vary the number of attention heads and the attention key and value dimensions,
 
---
